In [1]:
import json

path = "data\\corpus_law_pub.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)
    laws = []
    for item in data:
        # Xử lý dữ liệu ở đây
        law_id = item.get("law_id", "N/A")
        articles = item.get("content", [])
        for article in articles:
            article_id = article.get("article_id", "N/A")
            article_content = article.get("content", "")
            laws.append(
                {"law_id": law_id, "article_id": article_id, "content": article_content}
            )

    print(f"\nTổng số luật: {len(laws)}")


Tổng số luật: 3352


In [2]:
import pandas as pd

df = pd.read_parquet("data\\law_embeddings.parquet")

df.head()

,law_id,law_db_id,aid,text,char_count,word_count,token_count,embedding
0,47/2010/QH12,9,270,"Luật này quy định về việc thành lập, tổ chức, ...",287,60,71.75,"[0.005203068722039461, 0.018226539716124535, -..."
1,47/2010/QH12,9,271,Luật này áp dụng đối với các đối tượng sau đây...,510,109,127.50,"[0.009045800194144249, 0.030976060777902603, -..."
2,47/2010/QH12,9,272,"1. Việc thành lập, tổ chức và hoạt động, kiểm ...",1161,251,290.25,"[0.02283167466521263, 0.01048367004841566, -0...."
3,47/2010/QH12,9,273,"Trong Luật này, các từ ngữ dưới đây được hiểu ...",10236,2264,2559.00,"[-0.004261949565261602, 0.0008975652744993567,..."
4,47/2010/QH12,9,274,Tổ chức không phải là tổ chức tín dụng không đ...,456,99,114.00,"[0.00746202701702714, 0.0022314542438834906, -..."


In [4]:
len(df['embedding'][0])

1024

In [5]:
df.columns

Index(['law_id', 'law_db_id', 'aid', 'text', 'char_count', 'word_count',
       'token_count', 'embedding'],
      dtype='str')

In [6]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="law_corpus", metadata={"hnsw:space": "cosine"}
)

In [9]:
metadatas = [
    {
        "law_id": str(row["law_id"]),
        "law_db_id": str(row["law_db_id"]),
        "char_count": int(row["char_count"]),
        "word_count": int(row["word_count"]),
        "token_count": int(row["token_count"]),
    }
    for _, row in df.iterrows()
]

In [ ]:
collection.add(
    ids=df["aid"].astype(str).tolist(),
    embeddings=df["embedding"].tolist(),
    documents=df["text"].tolist(),
    metadatas=metadatas,
)

In [11]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_collection("law_corpus")

In [12]:
print(collection.count())   # số vector đã lưu
print(collection.name)

3280
law_corpus


In [13]:
results = collection.query(
    query_texts=["luật dân sự"],
    n_results=5
)

print(results)

C:\Users\Admin\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:33<00:00, 2.49MiB/s]


InvalidArgumentError: Collection expecting embedding with dimension of 1024, got 384

In [14]:
data = collection.get(
    limit=10
)

print(data)

{'ids': ['270', '271', '272', '273', '274', '275', '276', '277', '278', '279'], 'embeddings': None, 'documents': ['Luật này quy định về việc thành lập, tổ chức, hoạt động, kiểm soát đặc biệt, tổ chức lại, giải thể tổ chức tín dụng; việc thành lập, tổ chức, hoạt động của chi nhánh ngân hàng nước ngoài, văn phòng đại diện của tổ chức tín dụng nước ngoài, tổ chức nước ngoài khác có hoạt động ngân hàng.', 'Luật này áp dụng đối với các đối tượng sau đây: 1. Tổ chức tín dụng; 2. Chi nhánh ngân hàng nước ngoài; 3. Văn phòng đại diện của tổ chức tín dụng nước ngoài, tổ chức nước ngoài khác có hoạt động ngân hàng; 4. Tổ chức, cá nhân có liên quan đến việc thành lập, tổ chức, hoạt động, kiểm soát đặc biệt, tổ chức lại, giải thể tổ chức tín dụng; việc thành lập, tổ chức, hoạt động của chi nhánh ngân hàng nước ngoài, văn phòng đại diện của tổ chức tín dụng nước ngoài, tổ chức nước ngoài khác có hoạt động ngân hàng.', '1. Việc thành lập, tổ chức và hoạt động, kiểm soát đặc biệt, tổ chức lại, giải t

In [21]:
data = collection.get(include=["documents", "metadatas", "embeddings"])
len(data["embeddings"][0])

1024